<h2> Instruction</h2>

this code extracts data from the zip file. the zip file and this code file must be in the same directory.

- Before executing this code, download the ZIP file **FFAEXT.ZIP**.
- Move the ZIP file to the same folder as the code file (**.py** / **.ipynb**).
- Finally, run the code.
- the output excel file is in the form : **FFAEXT_date hour.xlsx**

- New function
Add PDF output @ Jingyi Wu


In [ ]:
#------------------------------------------------ Ensure required libs are installed ----------------------------------------
import importlib
import subprocess
import sys

def ensure_pip_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])

# reportlab package provides reportlab.lib and reportlab.pdfgen
ensure_pip_package("reportlab", "reportlab")



#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from time import sleep
import datetime
from pandas import ExcelWriter
import os
from zipfile import ZipFile
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'GB FCAUK' ## change to current controller name

print(f"[INFO] : Running {regulatorName} Web Scraping Tool v.1.2 | Last update : 06/08/2024")
now=datetime.datetime.now()
filename = '{}_{}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])
scriptfolder=os.getcwd()
os.chdir(scriptfolder)
writer = ExcelWriter(filename, engine='openpyxl')
csvdict = {'ID': [], 'colum0': [], 'Name': [], 'RegulationType': [], 'City': [], 'phone': [], 'Address1': [], 'Address2': []}
# sqldict = {'Name': [], 'Registered company number': [], 'Firm reference number': [], 'RegulationType': [], 'City': [], 'BP': [], 'Address1': [], 'Address2': [], 'phone': []}

sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
		  'Phone - Mother company': [], 'Check': []}

RegType = {
"Authorised"                : "Authorised",
"No longer authorised"      : "No longer authorised" ,
"No longer registered as a" : "No longer registered as an appointed Representative" ,
"Authorised - applied to c" : "Authorised - applied to cancel" ,
"Revoked"                   : "Revoked" ,
"Authorised - in special a" : "Authorised - in special administration" ,
"Appointed representative"  : "Appointed representative" ,
"Authorised - in liquidati" : "Authorised - in liquidation" ,
"Cancelled"                 : "Cancelled" ,
"EEA Authorised - Contract" : "EEA Authorised - Contractual run-off" ,
"EEA Authorised - Supervis" : "EEA Authorised - Supervised run-off" ,
"Authorised - in administr" : "Authorised - in administration" ,
"Authorised Schedule 5 - O" : "Authorised Schedule 5 - Operator/depositary/trustee of a temporary recognised scheme" ,
"Registered"                : "Registered" ,
"EEA Authorised - Former p" : "EEA Authorised - Former passporting firm" ,
"EEA Authorised"            : "EEA Authorised" ,
"EEA Authorised - Applied"  : "EEA Authorised - applied to cancel" ,

"Authorised Schedule 5"     : "Authorised Schedule 5 - Operator/depositary/truste" ,

"Authorised - Closed to ne" : "Authorised - Closed to new business" ,
"Lapsed"                    : "Lapsed" ,
"Temporary Permission"      : "Temporary Permission" ,
"Registered - Former"       : "Registered - Former" ,
"Certified"                 : "Certified" ,
"Authorised - Closed to Re" : "Authorised - Closed to Regulated Business" ,
}
reg = 'GB FCAUK 1'
RegulationType_Revoked = ['No longer authorised', 'No longer registered as an appointed Representative', 'Revoked', 'Appointed representative', 'Cancelled', 'Lapsed']
processdate = now.strftime('%Y-%m-%d')
#------------------------------------------------ Begin_Fouction ----------------------------------------
def extract_zipFile(zipfile, select_file, dst):
    with ZipFile(zipfile, 'r') as zipObj:
        FileNames = zipObj.namelist()
        for fileName in FileNames:
            if fileName == select_file:
                zipObj.extract(fileName, dst)

def bourange_same_length_array(sqldict) :
    maxlen = len(sqldict['ListProcessDate'])
    for key, val in sqldict.items():
        if len(sqldict[key]) != maxlen:
            empty = []
            total_empty = maxlen - len(sqldict[key])
            for i in range(total_empty):
                empty.append('')
            sqldict[key]=sqldict[key]+empty
    return sqldict
#------------------------------------------------ Begin_Main ---------------------------------------- 
file = os.path.join(scriptfolder, list(filter(lambda x:x.endswith(".ZIP"), os.listdir(scriptfolder)))[0] )
filePath = os.path.join(scriptfolder, file)
print(f'[INFO] : Extraction "FFAEXT" ')
extract_zipFile(filePath, "FFAEXT", scriptfolder)
csv_file = os.path.join(scriptfolder, list(filter(lambda x:x.endswith("FFAEXT"), os.listdir(scriptfolder)))[0] )

with open(csv_file, "rb") as myfile:
    print(f'[INFO] : Data structuring "FFAEXT" ...')
    for line in myfile:
        row = line.decode(errors='ignore').replace('\r\n', '')
        name = row[19:69].strip()
        # if not name.isdigit() and len(name)>1 :
        csvdict['ID'].append(row[:18].strip())
        csvdict['Name'].append(name)
        csvdict['colum0'].append(row[69:94].strip())
        csvdict['RegulationType'].append(row[94:119].strip())
        csvdict['City'].append(row[119:144].strip())
        csvdict['phone'].append(row[144:169].strip())
        csvdict['Address1'].append(row[169:206].strip())
        csvdict['Address2'].append(row[206:].strip())

df=pd.DataFrame(csvdict)
print(f'[INFO] : wait, {len(df)} data processing ...')
for items in range(0,len(df),3):

    indicatif = df['Address2'].iloc[items+1][-3:]
    if  not indicatif[-2:].isdigit() :
        indicatif = ''
    
    addr2 = str(df['Address2'].iloc[items] +', ' + df['Address1'].iloc[items+1]).strip()
    if len(addr2)>1:
        if addr2[0] == ',':
            addr2 = addr2[1:]
        if addr2[-1] == ',':
            addr2 = addr2[:-1]
    addr2 = '' if addr2 ==',' else addr2.strip()

    addr1 =  (df['phone'].iloc[items] + df['Address1'].iloc[items]).strip()
    try:
        if addr1.find('N') != -1:
            set = 'N'
        if addr1.find('X') != -1:
            set = 'X'
        addr1 = addr1.split(set)[-1]
    except:
        pass

    try:
        RegulationType = RegType[df['RegulationType'].iloc[items+2].strip()]
    except:
        RegulationType = df['RegulationType'].iloc[items+2]
    
    sqldict['Name'].append(df['Name'].iloc[items])
    sqldict['InternalID_1_type'].append('Registered company number')
    sqldict['InternalID_1'].append(df['ID'].iloc[items][4:-6])
    sqldict['InternalID_2_type'].append('Firm reference number')
    sqldict['InternalID_2'].append(df['ID'].iloc[items][-6:])

    sqldict['RegulationType'].append(RegulationType)
    sqldict['City'].append(df['City'].iloc[items+1])
    sqldict['Address_2'].append(addr2)
    sqldict['Address_1'].append(addr1)
    sqldict['Zip'].append(df['Address2'].iloc[items+1][:-3])
    sqldict['Phone'].append(str(indicatif+' '+ df['Name'].iloc[items+2]).strip())

    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegCtry'].append(reg.split(' ')[0])
    sqldict['RegCode'].append(reg.split(' ')[1])
    sqldict['ListCode'].append(reg.split(' ')[-1])


    # if items == 1000:
    #     break

sqldict = bourange_same_length_array(sqldict)
os.remove(csv_file)
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df_sql = pd.DataFrame(sqldict)

# drop duplicates and Revoked company
for Reg_Revoked in RegulationType_Revoked :
    df_sql =  df_sql.drop(df_sql[df_sql.RegulationType == Reg_Revoked].index) 

df_sql = df_sql.drop_duplicates(subset = ["Name", "Address_1"], keep = 'first')
df_sql = df_sql.reset_index(drop=True)

print(f'[INFO] : wait, save data ({len(df_sql)} company) to excel file ...')

# Split DataFrame into chunks and save each chunk to a separate sheet
chunk_size = 1000000  # Adjust chunk size as needed
num_chunks = len(df_sql) // chunk_size + 1

for i in range(num_chunks):
    start_row = i * chunk_size
    end_row = (i + 1) * chunk_size
    chunk_df = df_sql.iloc[start_row:end_row]
    sheet_name = f'SQL Ready Part {i+1}'
    chunk_df.to_excel(writer, sheet_name, index=False)

#writer.save()
writer.close()
sleep(2)
print('[INFO] : Finish | OK')

print('[INFO] : PDF generation')
def export_list_to_pdf(data_list, pdf_filename):
    c = canvas.Canvas(pdf_filename, pagesize=letter)
    c.setFont("Helvetica", 12)
    x = 72
    y = 720
    max_lines_per_page = 33 # Maximum number of lines per page
    line_count = 0
    
    # Add each item from the list to the PDF
    for item in data_list:
        c.drawString(x, y, str(item))
        y -= 20  # Move to the next line
        line_count += 1
        if line_count >= max_lines_per_page: # Check if we need to add a new page
            c.showPage()  # Add a new page
            c.setFont("Helvetica", 12)  # Reset font
            x = 72  # Reset x position
            y = 720  # Reset y position
            line_count = 0  # Reset line counter
            
    c.save()
regulatorName = 'GB_FCAUK_FFAXT_List_1 '
filename = '{} Ready {}'.format(regulatorName, str(now).replace(":",".")[:-7])
export_list_to_pdf(df_sql['Name'].tolist(), f'{filename}.pdf')

[INFO] : Running GB FCAUK Web Scraping Tool v.1.2 | Last update : 06/08/2024
[INFO] : Extraction "FFAEXT" 
[INFO] : Data structuring "FFAEXT" ...
[INFO] : wait, 858141 data processing ...
[INFO] : wait, save data (37024 company) to excel file ...


C:\Users\wuj1\AppData\Local\Temp\4\ipykernel_24584\633239221.py:177: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  chunk_df.to_excel(writer, sheet_name, index=False)


[INFO] : Finish | OK
[INFO] : PDF generation
